In [3]:
# 1. تثبيت المكتبات المطلوبة للواجهة والتفسير
!pip install -q gradio grad-cam huggingface_hub

import gradio as gr
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
from PIL import Image
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from huggingface_hub import hf_hub_download, login


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. إعدادات الفئات والأسماء
# ==========================================
skin_classes = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
skin_desc = {
    'akiec': 'Actinic Keratoses (محتمل التسرطن)',
    'bcc': 'Basal Cell Carcinoma (سرطان الخلايا القاعدية)',
    'bkl': 'Benign Keratosis (تقران حميد)',
    'df': 'Dermatofibroma (ورم ليفي جلدي)',
    'mel': 'Melanoma (ميلانوما - سرطان جلدي خطير)',
    'nv': 'Melanocytic Nevi (شامة طبيعية حميدة)',
    'vasc': 'Vascular Lesion (آفة وعائية)'
}

xray_classes = ['Normal (طبيعي)', 'Pneumonia (التهاب رئوي)']
xray_desc = {c: c for c in xray_classes}

# ==========================================
# 3. دوال بناء النماذج وتحميل الأوزان
# ==========================================
def load_base_model(model_name, num_classes, repo_id, filename):
    if model_name == "resnet50":
        model = models.resnet50()
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "densenet121":
        model = models.densenet121()
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0()
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        
    path = hf_hub_download(repo_id=repo_id, filename=filename)
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

print("جاري سحب النماذج الطبية من السحابة، يرجى الانتظار...")

# مستودعات Hugging Face الخاصة بك (يرجى التأكد من اسم مستودع الأشعة السينية)
skin_repo = "talalsvu/skin-cancer-base-models"
xray_repo = "talalsvu/chest-xray-base-models" 

# تحميل نماذج الأمراض الجلدية
skin_models = [
    load_base_model("resnet50", 7, skin_repo, "resnet50_skin_best_weights.pth"),
    load_base_model("densenet121", 7, skin_repo, "densenet121_skin_best_weights.pth"),
    load_base_model("efficientnet_b0", 7, skin_repo, "efficientnet_b0_skin_best_weights.pth")
]

# تحميل نماذج الأشعة السينية (تأكد من مطابقة أسماء الملفات لما رفعته سابقاً)
xray_models = [
    load_base_model("resnet50", 2, xray_repo, "resnet50_best_weights.pth"),
    load_base_model("densenet121", 2, xray_repo, "densenet121_best_weights.pth"),
    load_base_model("efficientnet_b0", 2, xray_repo, "efficientnet_b0_best_weights.pth")
]

print("تم تجهيز جميع النماذج بنجاح!")

# ==========================================
# 4. خوارزمية التجميع الديناميكي والتشخيص
# ==========================================
def calculate_entropy(probs):
    return -torch.sum(probs * torch.log(probs + 1e-9), dim=1)

def process_image(img):
    img_resized = img.resize((224, 224)).convert('RGB')
    img_array = np.array(img_resized, dtype=np.float32) / 255.0
    img_tensor = np.transpose(img_array, (2, 0, 1))
    img_tensor = torch.tensor(img_tensor).unsqueeze(0).to(device)
    return img_tensor, img_array

def predict_hybrid(img, task="skin"):
    img_tensor, img_array = process_image(img)
    
    models_list = skin_models if task == "skin" else xray_models
    classes = skin_classes if task == "skin" else xray_classes
    desc = skin_desc if task == "skin" else xray_desc
    gradcam_model = models_list[0] # استخدام ResNet-50 للتفسير
        
    with torch.no_grad():
        probs_list = [F.softmax(model(img_tensor), dim=1) for model in models_list]
        all_probs = torch.stack(probs_list)
        
        # تطبيق التجميع الديناميكي
        entropies = torch.stack([calculate_entropy(p) for p in probs_list])
        competences = 1.0 / (entropies + 1e-6)
        weights = competences / torch.sum(competences, dim=0)
        weights = weights.unsqueeze(-1)
        
        final_probs = torch.sum(all_probs * weights, dim=0)[0].cpu().numpy()
        
    predicted_idx = np.argmax(final_probs)
    results_dict = {f"{desc[classes[i]]}": float(final_probs[i]) for i in range(len(classes))}
        
    # توليد الخريطة الحرارية (يتم خارج torch.no_grad() لأنها تحتاج التدرجات)
    target_layers = [gradcam_model.layer4[-1]]
    cam = GradCAM(model=gradcam_model, target_layers=target_layers)
    targets = [ClassifierOutputTarget(predicted_idx)]
    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)[0, :]
    visualization = show_cam_on_image(img_array, grayscale_cam, use_rgb=True)
    
    return results_dict, visualization

# ==========================================
# 5. بناء واجهة الويب (Gradio) بثلاث تبويبات
# ==========================================
with gr.Blocks(theme=gr.themes.Base()) as interface:
    gr.Markdown("# 🏥 النظام الطبي الهجين للتشخيص الذكي (Dynamic Ensemble AI)")
    gr.Markdown("### 🎓 نموذج أولي (Prototype) مطور لرسالة الماجستير في علوم الويب.")
    gr.Markdown("يعتمد هذا النظام على التجميع الديناميكي لثلاثة نماذج تعلم عميق (ResNet50, DenseNet121, EfficientNet-B0) موجهة بـ (الإنتروبيا) لتقديم تشخيص دقيق وقابل للتفسير.")
    
    with gr.Tabs():
        # التبويب الأول: الأمراض الجلدية
        with gr.TabItem("🔬 تشخيص الأمراض الجلدية (Skin Cancer)"):
            with gr.Row():
                with gr.Column():
                    skin_in = gr.Image(type="pil", label="قم برفع صورة الآفة الجلدية (Dermoscopy)")
                    skin_btn = gr.Button("تحليل الصورة الجلدية", variant="primary")
                with gr.Column():
                    skin_out_label = gr.Label(num_top_classes=3, label="القرار المدمج (أعلى 3 احتمالات)")
                    skin_out_img = gr.Image(label="خريطة التفسير الحرارية (Grad-CAM)")
            skin_btn.click(lambda img: predict_hybrid(img, "skin"), inputs=skin_in, outputs=[skin_out_label, skin_out_img])
            
        # التبويب الثاني: الأشعة السينية
        with gr.TabItem("🩻 تشخيص الأشعة السينية (Chest X-Ray)"):
            with gr.Row():
                with gr.Column():
                    xray_in = gr.Image(type="pil", label="قم برفع صورة الأشعة السينية للصدر")
                    xray_btn = gr.Button("تحليل صورة الأشعة", variant="primary")
                with gr.Column():
                    xray_out_label = gr.Label(label="القرار المدمج (التهاب رئوي أم طبيعي)")
                    xray_out_img = gr.Image(label="خريطة التفسير الحرارية (Grad-CAM)")
            xray_btn.click(lambda img: predict_hybrid(img, "xray"), inputs=xray_in, outputs=[xray_out_label, xray_out_img])

print("جاري إطلاق واجهة الويب التفاعلية...")
interface.launch(share=True)

Using device: cuda
جاري سحب النماذج الطبية من السحابة، يرجى الانتظار...
تم تجهيز جميع النماذج بنجاح!
جاري إطلاق واجهة الويب التفاعلية...


/tmp/ipykernel_55/2648029228.py:125: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Base()) as interface:


* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://4352fdc73fbf83ac7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
